In [0]:
%sql
SELECT * FROM bronze_dev.cms.hospital_general_information

In [0]:
%sql
SELECT UPPER(CountyParish), State FROM bronze_dev.cms.hospital_general_information
EXCEPT
SELECT
  county_name_no_suffix,
  states.state_abbreviation
FROM
  silver_dev.geo.counties AS counties
  LEFT JOIN silver_dev.geo.states AS states
  ON counties.state_fips = states.state_fips
ORDER BY 1

In [0]:
%sql
SELECT *
FROM
  silver_dev.geo.counties AS counties
WHERE county_name ilike '%kalb%'

In [0]:

%sql
SELECT LOWER(CountyParish), State FROM bronze_dev.cms.hospital_general_information
EXCEPT
SELECT
  REPLACE(LOWER(county_name), ' county', ''), 
  state_name 
FROM
  silver_dev.geo.counties

# Column EDA

In [0]:
%sql
SELECT DISTINCT Hospital_overall_rating
FROM bronze_dev.cms.hospital_general_information

# Python pipeline

In [0]:
import importlib
import utils.helpers

importlib.reload(utils.helpers)

In [0]:
df = spark.table("bronze_dev.cms.hospital_general_information")
df = utils.helpers.prep_silver_df(df)
display(df)

In [0]:
%sql
select * from silver_dev.geo.counties

## Join with counties

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

counties = spark.table("silver_dev.geo.counties")


df_joined = (
    df.alias("d")
      .join(
          broadcast(counties).alias("c"),
          (F.lower(F.trim(F.col("d.county_parish"))) == F.lower(F.trim(F.col("c.county_name_no_suffix")))) &
          (F.upper(F.trim(F.col("d.state"))) == F.upper(F.trim(F.col("c.state_abbreviation")))),
          "left"
      )
      .select(
          "d.*",
          F.col("c.county_fips")  # 👈 bring it in here
      )
)
df_joined.display()

### Survey columns to determine appropriate datatype

In [0]:
from pyspark.sql import functions as F

for col in df_joined.columns:
    distinct_count = df_joined.select(F.countDistinct(F.col(col))).collect()[0][0]
    
    print(f"\nColumn: {col}, Distinct count: {distinct_count}")
    
    if distinct_count < 20:
        
        values = (
            df_joined.select(col)
              .distinct()
              .orderBy(col)
              .collect()
        )
        
        # Extract values from Row objects
        values = [row[0] for row in values]
        
        print(f"Values: {values}")

In [0]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F

cols_to_clean = [
    "hospital_overall_rating",
    "mort_group_measure_count",
    "count_of_facility_mort_measures",
    "count_of_mort_measures_better",
    "count_of_mort_measures_no_different",
    "count_of_mort_measures_worse",
    "safety_group_measure_count",
    "count_of_facility_safety_measures",
    "count_of_safety_measures_better",
    "count_of_safety_measures_no_different",
    "count_of_safety_measures_worse",
    "readm_group_measure_count",
    "count_of_facility_readm_measures",
    "count_of_readm_measures_better",
    "count_of_readm_measures_no_different",
    "count_of_readm_measures_worse",
    "pt_exp_group_measure_count",
    "count_of_facility_pt_exp_measures",
    "te_group_measure_count",
    "count_of_facility_te_measures",
]

for c in cols_to_clean:
    print(f"Column: {c}")
    df_joined = df_joined.withColumn(
        c,
        F.when(F.col(c) == "Not Available", None)
         .otherwise(F.col(c))
         .cast("int")
    )
display(df_joined)

In [0]:
df_joined.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("silver_dev.healthcare.hospitals")

In [0]:
%sql
SELECT * FROM silver_dev.healthcare.hospitals